# 02 — Model Trustworthiness & Calibration Analysis

This notebook provides clinical calibration analysis:
1. Expected Calibration Error (ECE) computation across Cross-Entropy, Focal, and CORAL models
2. Reliability diagrams visualizing confidence vs. observed empirical accuracy
3. Post-hoc Temperature Scaling calibration optimization via L-BFGS
4. Nuance: Adjacent-categories formulation projecting CORAL sigmoid outputs to categorical distributions

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from src.evaluation.calibration import TemperatureScaler, plot_reliability_diagram
from src.evaluation.metrics import compute_ece
from src.losses.coral import coral_probabilities
print("Calibration suite loaded.")

### Synthetic Demo: Pre- and Post-Calibration Comparison

In [ ]:
# Generate mock overconfident logits
torch.manual_seed(42)
mock_logits = torch.randn(200, 4) * 3.5  # overconfident CORAL logits
mock_labels = torch.randint(0, 5, (200,))

scaler = TemperatureScaler()
opt_temp = scaler.fit(mock_logits, mock_labels, is_ordinal=True)
print(f"Fitted Temperature T: {opt_temp:.3f}")

probs_uncal = coral_probabilities(mock_logits)
probs_cal = coral_probabilities(scaler(mock_logits))

fig = plot_reliability_diagram(probs_cal, mock_labels, title="Post-Calibration Reliability Diagram")
plt.show()